# 24-modulation training on Colab's GPU

Trains the full 24-class version of the model on a free Colab GPU, using more data
than the laptop CPU run could handle. Nothing here changes the model — TensorFlow
just runs the same scripts far faster on the GPU.

**Before you run anything:**
1. Upload a folder named `5G-Colab` to your Google Drive containing:
   ```
   5G-Colab/
     Data/signals.npy
     Data/labels.npy
     Data/classes.txt
     archive (1)/snrs.npy
     scripts/            <- all our .py files
   ```
2. Turn the GPU on: menu **Runtime -> Change runtime type -> Hardware accelerator = GPU**.

Then run the cells top to bottom.

## 1. Check we actually got a GPU
If this errors or shows no GPU, redo Runtime -> Change runtime type.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive
A pop-up will ask permission — that lets Colab read the folder you uploaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/5G-Colab'
import os
assert os.path.isdir(DRIVE), f'Did not find {DRIVE} — check the folder name/location in Drive.'
print('Found:', sorted(os.listdir(DRIVE)))

## 3. Copy the data to Colab's fast local disk
Reading the 19.5 GB file straight off Drive is slow for the random sampling
`data_prep` does, so we copy it to the local disk once (a few minutes). We recreate
the same project layout the scripts expect (`Data/`, `archive (1)/`, `scripts/`).

In [ ]:
import shutil, os, time

ROOT = '/content/proj'
os.makedirs(f'{ROOT}/Data', exist_ok=True)
os.makedirs(f'{ROOT}/archive (1)', exist_ok=True)

# scripts (tiny)
if os.path.isdir(f'{ROOT}/scripts'):
    shutil.rmtree(f'{ROOT}/scripts')
shutil.copytree(f'{DRIVE}/scripts', f'{ROOT}/scripts')

# data files (the big copy)
to_copy = [
    (f'{DRIVE}/Data/signals.npy',        f'{ROOT}/Data/signals.npy'),
    (f'{DRIVE}/Data/labels.npy',         f'{ROOT}/Data/labels.npy'),
    (f'{DRIVE}/Data/classes.txt',        f'{ROOT}/Data/classes.txt'),
    (f'{DRIVE}/archive (1)/snrs.npy',    f'{ROOT}/archive (1)/snrs.npy'),
]
for src, dst in to_copy:
    t = time.time()
    print(f'copying {os.path.basename(src)} ...', end=' ', flush=True)
    shutil.copy(src, dst)
    print(f'{os.path.getsize(dst)/1e9:.1f} GB in {time.time()-t:.0f}s')
print('Done.')

## 4. Prepare a 24-class sample
`--all-24` uses every class; `--per-class` sets how many examples of each.

**20,000/class (480k total)** is safe on a free GPU runtime. To go bigger (30–40k),
switch to a **High-RAM** runtime first (Runtime -> Change runtime type), otherwise the
sampling step can run out of memory.

In [ ]:
PER_CLASS = 20000  # bump to 30000-40000 only on a High-RAM runtime
!cd {ROOT} && python scripts/data_prep.py --all-24 --per-class {PER_CLASS}

## 5. Train both models (VGG and ResNet)
Same scripts as the laptop, now on the GPU — a few minutes each, not an hour.
Both train on the exact same prepared data from step 4, one after the other.

In [ ]:
!cd {ROOT} && python scripts/train.py --model vgg
!cd {ROOT} && python scripts/train.py --model resnet

## 6. Evaluate both + high-SNR score
Confusion matrix and per-class precision/recall for each model, then the paper-style
clean-signal number for both side by side (`high_snr.py` scores both at once).

In [ ]:
!cd {ROOT} && python scripts/evaluate.py --model vgg
!cd {ROOT} && python scripts/evaluate.py --model resnet
!cd {ROOT} && python scripts/high_snr.py

## 7. Save the results back to Drive
Colab wipes local disk when the session ends, so we copy the trained model, the
prepared arrays, and the figures into `5G-Colab/outputs/` on your Drive to keep them.

In [ ]:
import shutil, os
OUT = f'{DRIVE}/outputs'
for sub in ['results', 'models', 'prepared']:
    src = f'{ROOT}/{sub}'
    if os.path.isdir(src):
        dst = f'{OUT}/{sub}'
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print('saved', dst)
print('All results are in your Drive under 5G-Colab/outputs/.')